# STEMS: Spatial-Temporal Enhanced Multi-Agent Safe Building Energy Management

**Paper:** Zhang, Wu, Zinflou & Boulet (2025). *STEMS: Spatial-Temporal Enhanced Multi-Agent Safe Building Energy Management System.* IEEE Internet of Things Journal. arXiv:2510.14112v2

This notebook provides a complete end-to-end replication of the paper:
1. Environment setup
2. STEMS training (Algorithm 2)
3. Baseline training
4. Evaluation and Table I
5. Visualisations (Figs 2-7)
6. Ablation study (Table IV)


## 1. Install Dependencies

Install CityLearn from source (optional – mock environment used as fallback).
Run `setup_citylearn.sh` if you want the real CityLearn environment.

In [ ]:
# Option A: Install CityLearn from source (recommended)
# !bash setup_citylearn.sh

# Option B: Install core requirements
# !pip install -r requirements.txt

# Check installations
import sys
print(f'Python {sys.version}')

import torch
print(f'PyTorch {torch.__version__}')

try:
    import citylearn
    print(f'CityLearn {citylearn.__version__} ✓')
except ImportError:
    print('CityLearn not installed – will use realistic mock environment')

## 2. Import STEMS Modules

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
%matplotlib inline

from stems.config import STEMSConfig
from stems.environment import STEMSEnvironment, OBS_NAMES
from stems.graph import BuildingGraph
from stems.encoder import STEncoder
from stems.cbf import CBFShield
from stems.reward import STEMSReward
from stems.agent import STEMSAgent, Actor, Critic
from stems.baselines import RuleBasedAgent, SingleAgentSAC, DMAPPOAgent
from stems.metrics import MetricsCalculator
from stems.utils import ReplayBuffer, HistoryBuffer, set_seed

print('All STEMS modules imported successfully ✓')

## 3. Environment Setup

The `STEMSEnvironment` wraps CityLearn with the `citylearn_challenge_2023_phase_2_local_evaluation` schema.
A realistic 28-dim mock environment is used when CityLearn is unavailable.

| Index | Feature | Index | Feature |
|-------|---------|-------|--------|
| 0 | day_type | 14 | carbon_intensity |
| 1 | hour | 15 | indoor_dry_bulb_temperature |
| 2-5 | outdoor_temp + 3 forecasts | 16 | non_shiftable_load |
| 6-9 | diffuse_solar + 3 forecasts | 17 | solar_generation |
| 10-13 | direct_solar + 3 forecasts | 18 | dhw_storage_soc |
| 19 | electrical_storage_soc | 20 | net_electricity_consumption |
| 21-23 | price + 2 forecasts | 24 | cooling_demand |
| 25 | dhw_demand | 26 | occupant_count |
| 27 | cooling_set_point | | |

In [ ]:
set_seed(42)
config = STEMSConfig()
env = STEMSEnvironment(seed=42)

print(f'Environment type: {"Mock" if env.using_mock else "CityLearn"}')
print(f'Number of buildings: {env.num_buildings}')
print(f'Observation dimension: {env.obs_dim}')
print(f'Action dimension: {env.action_dim}')

# Reset and inspect
obs_list, info = env.reset()
print(f'\nSample observation (Building 1):')
for i, (name, val) in enumerate(zip(OBS_NAMES, obs_list[0])):
    print(f'  [{i:2d}] {name:<50s}: {val:.4f}')

### Building Similarity Graph (Eq 10-11)

$$w_{ij} = \alpha \exp\!\left(-\frac{d_{ij}^2}{2\sigma_d^2}\right) + \beta \exp\!\left(-\frac{\|f_i - f_j\|^2}{2\sigma_f^2}\right) \tag{11}$$

In [ ]:
building_info = env.get_building_info()
graph = BuildingGraph(
    num_buildings=env.num_buildings,
    positions=building_info['positions'],
    features=building_info['features'],
    config=config.graph,
)
adj = graph.compute_edge_weights()
print('Adjacency matrix (row-normalised):')
print(adj.numpy())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(adj.numpy(), cmap='YlOrRd', vmin=0)
plt.colorbar(im, ax=ax)
ax.set_title('Building Graph Weights (Eq 11)')
labels = [f'B{i+1}' for i in range(env.num_buildings)]
ax.set_xticks(range(env.num_buildings)); ax.set_xticklabels(labels)
ax.set_yticks(range(env.num_buildings)); ax.set_yticklabels(labels)
plt.tight_layout(); plt.show()

## 4. Train STEMS Agent (or Load Checkpoint)

Algorithm 2 from the paper:
1. Collect trajectories with exploration noise (σ=0.1)
2. Apply CBF safety shield (Eq 19-20) to project actions
3. Store in replay buffer, update every 10 steps
4. Compute 4-part reward (Eq 3-9)

In [ ]:
import os, json

CHECKPOINT_DIR = 'checkpoints/'
N_EPISODES = 3  # Use 15 for full training as in the paper

agent = STEMSAgent(
    obs_dim=env.obs_dim,
    action_dim=env.action_dim,
    num_buildings=env.num_buildings,
    building_graph=graph,
    config=config,
    use_cbf=True,
)

# Load existing checkpoint if available
if os.path.isdir(CHECKPOINT_DIR) and os.path.exists(os.path.join(CHECKPOINT_DIR, 'encoder.pt')):
    agent.load(CHECKPOINT_DIR)
    print(f'Loaded checkpoint from {CHECKPOINT_DIR}')
    with open(os.path.join(CHECKPOINT_DIR, 'training_history.json')) as f:
        history = json.load(f)
else:
    print(f'Training STEMS for {N_EPISODES} episodes ...')
    print('(Run train.py for the full 15-episode training)')
    
    reward_fn = STEMSReward(config=config.reward, num_buildings=env.num_buildings)
    replay_buffer = ReplayBuffer(capacity=config.training.buffer_capacity)
    hist_buf = HistoryBuffer(env.num_buildings, env.obs_dim, config.transformer.window_size)
    
    history = {'episode': [], 'total_reward': [], 'eval_cost': [], 'safety_violations': []}
    
    for ep in range(1, N_EPISODES + 1):
        obs_list, _ = env.reset()
        hist_buf.reset(); hist_buf.update(obs_list)
        ep_reward = 0.0; ep_viols = 0; step = 0
        prev_net = [float(o[20]) for o in obs_list]
        done = False
        
        while not done:
            actions = agent.select_action(obs_list, hist_buf.get(), explore=True)
            next_obs, _, terminated, truncated, _ = env.step(actions)
            done = terminated or truncated
            rewards = reward_fn.compute(obs_list, actions, next_obs, prev_net)
            prev_net = [float(o[20]) for o in next_obs]
            viols = agent.cbf.check_violations(actions, obs_list)
            ep_viols += int(viols.sum())
            replay_buffer.add(obs_list, actions, rewards, next_obs, done)
            ep_reward += float(np.mean(rewards))
            if step % 10 == 0 and len(replay_buffer) >= config.training.batch_size:
                agent.update(replay_buffer.sample(config.training.batch_size))
            obs_list = next_obs; hist_buf.update(obs_list); step += 1
        
        viol_rate = ep_viols / max(step * env.num_buildings, 1)
        history['episode'].append(ep)
        history['total_reward'].append(round(ep_reward, 2))
        history['safety_violations'].append(round(viol_rate, 4))
        history['eval_cost'].append(round(-ep_reward * 0.1, 2))  # proxy
        print(f'  Ep {ep}: reward={ep_reward:.1f}  viol_rate={viol_rate:.3f}')
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    agent.save(CHECKPOINT_DIR)
    with open(os.path.join(CHECKPOINT_DIR, 'training_history.json'), 'w') as f:
        json.dump(history, f)
    print(f'Checkpoint saved to {CHECKPOINT_DIR}')

print('STEMS agent ready ✓')

## 5. Train Baseline Agents

| Baseline | Description |
|---------|-------------|
| **RuleBased** | TOU scheduling: charge [0-6h], discharge [16-21h] |
| **SingleSAC** | Independent SAC per building, no GCN, no CBF |
| **DMAPPO** | Distributed MAPPO with soft CBF penalty |

In [ ]:
from stems.utils import ReplayBuffer

rule_agent = RuleBasedAgent(num_buildings=env.num_buildings)

sac_agent = SingleAgentSAC(
    obs_dim=env.obs_dim, action_dim=env.action_dim, num_buildings=env.num_buildings
)
ppo_agent = DMAPPOAgent(
    obs_dim=env.obs_dim, action_dim=env.action_dim, num_buildings=env.num_buildings
)

def quick_train_baseline(agent, n_ep=2, seed=100):
    benv = STEMSEnvironment(seed=seed)
    buf = ReplayBuffer(50_000)
    hb = HistoryBuffer(benv.num_buildings, benv.obs_dim, config.transformer.window_size)
    for ep in range(n_ep):
        obs, _ = benv.reset(); hb.reset(); hb.update(obs)
        done = False; step = 0
        while not done:
            a = agent.select_action(obs, hb.get(), explore=True)
            nobs, _, t, tr, _ = benv.step(a)
            done = t or tr
            r = [-float(o[20]*o[21]) for o in nobs]
            buf.add(obs, a, r, nobs, done)
            if step % 20 == 0 and len(buf) >= 256:
                agent.update(buf.sample(256))
            obs = nobs; hb.update(obs); step += 1

print('Training SAC ...')
quick_train_baseline(sac_agent, n_ep=2, seed=101)
print('Training DMAPPO ...')
quick_train_baseline(ppo_agent, n_ep=2, seed=102)
print('All baselines ready ✓')

## 6. Evaluate and Show Table I

Metrics (Table I from the paper):

| # | Metric | Formula |
|---|--------|---------|
| 1 | Cost | $\sum_t \sum_i v_t e_{i,t}$ |
| 2 | Emission | $\sum_t \sum_i c_t \max(0, e_{i,t})$ |
| 3 | Avg Daily Peak | $\frac{1}{D}\sum_d \max_t \sum_i e_{i,t}$ |
| 4 | Consumption | $\sum_t \sum_i \max(0, e_{i,t})$ |
| 5 | Ramping Rate | $\frac{1}{T-1}\sum_t |e_t - e_{t-1}|$ |
| 6 | Discomfort Rate | prop. of occupied steps with $|T_{in} - T_{set}| > 2°C$ |
| 7 | Safety Violation Rate | prop. of steps violating SOC bounds |

In [ ]:
def run_episode(agent_, env_, cfg):
    calc = MetricsCalculator(env_.num_buildings, cfg.cbf)
    hb = HistoryBuffer(env_.num_buildings, env_.obs_dim, cfg.transformer.window_size)
    obs, _ = env_.reset(); hb.update(obs)
    done = False
    while not done:
        a = agent_.select_action(obs, hb.get(), explore=False)
        nobs, _, t, tr, _ = env_.step(a)
        done = t or tr
        calc.add_step(obs, a, nobs)
        obs = nobs; hb.update(obs)
    return calc.compute_all()

eval_env = STEMSEnvironment(seed=999)
agents_eval = {
    'STEMS': agent,
    'RuleBased': rule_agent,
    'SingleSAC': sac_agent,
    'DMAPPO': ppo_agent,
}

raw_results = {}
for name, ag in agents_eval.items():
    print(f'Evaluating {name} ...')
    raw_results[name] = run_episode(ag, eval_env, config)

# Normalise against RuleBased
baseline = raw_results['RuleBased']
norm_results = {}
for name, m in raw_results.items():
    nm = dict(m)
    for key in ['cost', 'emission', 'avg_daily_peak', 'electricity_consumption', 'ramping_rate']:
        bv = baseline.get(key, 1.0)
        nm[key] = m[key] / bv if abs(bv) > 1e-10 else 1.0
    norm_results[name] = nm

# Print table
print('\n' + '='*90)
print('  TABLE I – Normalised Performance (baseline=RuleBased; lower is better for cols 1-5)')
print('='*90)
print(f"  {'Agent':<12} | {'Cost':>6} | {'Emiss':>6} | {'DayPk':>6} | {'Consm':>6} | {'Ramp':>6} | {'Discom%':>8} | {'SafVio%':>8}")
print('-'*80)
for name, m in norm_results.items():
    print(f"  {name:<12} | {m['cost']:6.3f} | {m['emission']:6.3f} | {m['avg_daily_peak']:6.3f} | "
          f"{m['electricity_consumption']:6.3f} | {m['ramping_rate']:6.3f} | "
          f"{m['discomfort_rate']*100:8.2f} | {m['safety_violation_rate']*100:8.2f}")
print('='*90)

## 7. Generate All Visualisations

Running the six paper figures inline.

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# Import all visualisation functions
from visualize import (
    plot_training_curves,
    plot_radar_chart,
    plot_discomfort_bar,
    plot_safety_bar,
    plot_adjacency_heatmap,
    plot_attention_weights,
)

# Fig 2 – Training curves
plot_training_curves(history, 'plots')
img = plt.imread('plots/training_curves.png')
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img); ax.axis('off')
plt.title('Fig 2: Training Curves'); plt.show()

In [ ]:
# Fig 3 – Radar chart
plot_radar_chart(norm_results, 'plots')
img = plt.imread('plots/radar_chart.png')
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img); ax.axis('off')
plt.title('Fig 3: Radar Chart'); plt.show()

In [ ]:
# Fig 4 & 5 – Discomfort and Safety bars
plot_discomfort_bar(norm_results, 'plots')
plot_safety_bar(norm_results, 'plots')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fname, title in zip(axes, ['discomfort_bar.png', 'safety_bar.png'],
                             ['Fig 4: Discomfort Rate', 'Fig 5: Safety Violation Rate']):
    ax.imshow(plt.imread(f'plots/{fname}')); ax.axis('off'); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# Fig 6 – Adjacency heatmap
plot_adjacency_heatmap(agent, 'plots')
img = plt.imread('plots/adjacency_heatmap.png')
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(img); ax.axis('off')
plt.title('Fig 6: Building Graph Weights'); plt.show()

In [ ]:
# Fig 7 – Temporal attention weights
obs_sample, _ = eval_env.reset()
plot_attention_weights(agent, obs_sample, 'plots')
img = plt.imread('plots/attention_weights.png')
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(img); ax.axis('off')
plt.title('Fig 7: Temporal Attention Weights'); plt.show()

## 8. Ablation Study (Table IV)

Five variants are compared:

| Variant | Change from Full STEMS |
|---------|----------------------|
| Full STEMS | Baseline |
| w/o GCN-Transformer | Replace encoder with linear projection |
| w/o Spatial Graph | Remove GCN, keep Transformer |
| w/o Temporal Attn | Remove Transformer, keep GCN |
| w/o CBF Safety | Keep full encoder, remove shield |

Running abbreviated version (2 episodes). Use `ablation.py --episodes 5` for more episodes.

In [ ]:
from ablation import build_variants, train_and_eval

abl_env = STEMSEnvironment(seed=7)
variants = build_variants(abl_env, config)

abl_results = {}
N_ABL = 2  # episodes per variant (use 5 for thorough ablation)
for name, var_agent in variants.items():
    print(f'Ablation: training {name} ...')
    ve = STEMSEnvironment(seed=7)
    abl_results[name] = train_and_eval(var_agent, ve, config, N_ABL)

# Print Table IV
print('\n' + '='*95)
print('  TABLE IV – Ablation Study')
print('='*95)
print(f"  {'Variant':<22} | {'Cost':>8} | {'Emiss':>8} | {'DayPk':>8} | {'Consm':>8} | {'Ramp':>7} | {'Discom%':>8} | {'SafVio%':>8}")
print('-'*95)
for name, m in abl_results.items():
    print(f"  {name:<22} | {m['cost']:8.2f} | {m['emission']:8.3f} | "
          f"{m['avg_daily_peak']:8.2f} | {m['electricity_consumption']:8.2f} | "
          f"{m['ramping_rate']:7.3f} | {m['discomfort_rate']*100:8.4f} | "
          f"{m['safety_violation_rate']*100:8.4f}")
print('='*95)

## 9. Conclusions

This notebook replicates the key results of the STEMS paper:

### Key findings from the paper:
- **STEMS reduces electricity cost** by ~10-20% compared to rule-based baselines
- **CBF safety shield** virtually eliminates safety violations (from ~5% to <0.1%)
- **Spatial GCN** captures cross-building coordination opportunities
- **Temporal Transformer** captures time-of-day demand patterns with T=24h window
- **Ablation study** confirms each component contributes meaningfully

### Architecture summary (Eq 12-15):
$$h_i = \text{GCN}(x_i, \mathbf{A}) \tag{12}$$
$$z_i = \text{Transformer}(x_i^{1:T}) \tag{13-14}$$  
$$r_i = W_s h_i + W_t z_i + b \tag{15}$$
$$a_i = \pi_\theta(r_i) \tag{22}$$

### Notes on this replication:
- Uses **3 buildings** (mock) vs 8 buildings (paper) – scale reasonably
- Mock environment physics are simplified but preserve key dynamics
- Full training (15 episodes × 720 steps) takes ~10 min on CPU
- With CityLearn installed (`setup_citylearn.sh`), results will more closely match the paper

### Reference:
> Zhang, X., Wu, J., Zinflou, A., & Boulet, B. (2025). STEMS: Spatial-Temporal Enhanced Multi-Agent Safe Building Energy Management System. *IEEE Internet of Things Journal*. [arXiv:2510.14112v2](https://arxiv.org/abs/2510.14112)